# Duration-dependency diagnostic — cross-modality comparison

Compares how much of the apparent feature→depth signal is actually just a
proxy for segment duration, separately for **airborne** and **structure-borne**
feature sets.

For each modality this notebook computes:
1. Duration ↔ depth coupling strength
2. Raw Spearman correlation with depth for every feature
3. Partial Spearman correlation with depth **after controlling for duration**
4. Feature-level classification into *genuine depth signal* vs *duration proxy*

The key diagnostic is the cross-modality summary at the end:
- How many features survive duration partialling in each modality?
- Which feature families are most/least affected?
- Is one modality structurally more prone to duration confounding?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, rankdata, spearmanr

plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

# ── CONFIGURE ────────────────────────────────────────────────────────────────
# Point each path to the features CSV for that modality.
# Set to None to skip a modality.
AIRBORNE_CSV = None  # e.g. Path("data/features/airborne/features.csv")
STRUCTURE_CSV = None  # e.g. Path("data/features/structure/features_structure_v2.csv")

TARGET_COL = "depth_mm"
DURATION_COL = "duration_s"
PARTIAL_THRESH = 0.15  # |partial r| threshold for "genuine depth signal"

# Auto-detect project root (adjust if needed)
try:
    from vm_micro.utils.paths import PROJECT_ROOT
except ImportError:
    PROJECT_ROOT = Path(".")

# Fallback: uncomment and set manually if the above are None
# AIRBORNE_CSV  = PROJECT_ROOT / "data/features/airborne/features.csv"
# STRUCTURE_CSV = PROJECT_ROOT / "data/features/structure/features_structure_v2.csv"

META_COLS = {
    "modality",
    "record_name",
    "recording_root",
    "depth_mm",
    "step_idx",
    "duration_s",
    "sr_hz",
    "sr_hz_native",
    "sr_hz_used",
    "ds_rate",
    "file_path",
    "run_id",
    "batch_id",
    "slot",
    "mic_dx_mm",
    "mic_dy_mm",
    "mic_r_mm",
    "mic_log_r",
    "mic_angle_rad",
}
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
def partial_spearman(x, y, z):
    """Partial Spearman r between x and y, controlling for z.
    Returns (r_partial, p_value, n_used)."""
    tmp = pd.DataFrame({"x": x, "y": y, "z": z}).replace([np.inf, -np.inf], np.nan).dropna()
    n = len(tmp)
    if n < 4:
        return np.nan, np.nan, n
    if tmp["x"].nunique() < 2 or tmp["y"].nunique() < 2 or tmp["z"].nunique() < 2:
        return np.nan, np.nan, n
    xr = rankdata(tmp["x"].to_numpy()).astype(float)
    yr = rankdata(tmp["y"].to_numpy()).astype(float)
    zr = rankdata(tmp["z"].to_numpy()).astype(float)
    Z = np.column_stack([np.ones(n), zr])
    x_res = xr - Z @ np.linalg.lstsq(Z, xr, rcond=None)[0]
    y_res = yr - Z @ np.linalg.lstsq(Z, yr, rcond=None)[0]
    r, p = pearsonr(x_res, y_res)
    return float(r), float(p), n


def analyse_modality(df, modality_name):
    """Run the full duration-partialling analysis for one modality.
    Returns a DataFrame with one row per feature."""
    feat_cols = [
        c
        for c in df.columns
        if c not in META_COLS and c != TARGET_COL and pd.api.types.is_numeric_dtype(df[c])
    ]

    y = df[TARGET_COL]
    dur = df[DURATION_COL]
    dur_depth_r = float(spearmanr(dur, y, nan_policy="omit").statistic)

    rows = []
    for feat in feat_cols:
        raw_r = float(spearmanr(df[feat], y, nan_policy="omit").statistic)
        dur_r = float(spearmanr(df[feat], dur, nan_policy="omit").statistic)
        part_r, part_p, n_used = partial_spearman(df[feat], y, dur)

        family = feat.split("_")[0]
        rows.append(
            {
                "modality": modality_name,
                "feature": feat,
                "family": family,
                "raw_r": raw_r,
                "abs_raw_r": abs(raw_r),
                "r_duration": dur_r,
                "abs_r_duration": abs(dur_r),
                "partial_r": part_r,
                "abs_partial_r": abs(part_r) if pd.notna(part_r) else np.nan,
                "partial_p": part_p,
                "n_used": n_used,
            }
        )

    out = pd.DataFrame(rows)
    out["delta"] = out["abs_raw_r"] - out["abs_partial_r"]
    out["survived_015"] = out["abs_partial_r"] > PARTIAL_THRESH
    out["survived_020"] = out["abs_partial_r"] > 0.20
    out["dur_depth_r"] = dur_depth_r
    return out.sort_values("abs_partial_r", ascending=False).reset_index(drop=True)


print("Helper functions defined.")

## 1 · Load data and run analysis per modality

In [ ]:
results = {}

for label, csv_path in [("airborne", AIRBORNE_CSV), ("structure", STRUCTURE_CSV)]:
    if csv_path is None or not Path(csv_path).exists():
        print(f"⏭  Skipping {label} (path not set or file not found: {csv_path})")
        continue
    df = pd.read_csv(csv_path)
    print(f"\n{'=' * 60}")
    print(f"  {label.upper()}: {df.shape[0]} rows × {df.shape[1]} cols")
    dur_r = float(spearmanr(df[DURATION_COL], df[TARGET_COL], nan_policy="omit").statistic)
    print(f"  Duration ↔ depth Spearman r: {dur_r:.3f}")
    print(f"  Duration range: {df[DURATION_COL].min():.3f} – {df[DURATION_COL].max():.3f} s")
    print(f"{'=' * 60}")

    res = analyse_modality(df, label)
    results[label] = {"df": df, "partial": res}

    n_feat = len(res)
    n_015 = res["survived_015"].sum()
    n_020 = res["survived_020"].sum()
    best = res.iloc[0]
    print(f"  Features total:            {n_feat}")
    print(f"  |partial r| > 0.15:        {n_015}  ({100 * n_015 / n_feat:.1f}%)")
    print(f"  |partial r| > 0.20:        {n_020}  ({100 * n_020 / n_feat:.1f}%)")
    print(f"  Best partial feature:      {best['feature']}  |r|={best['abs_partial_r']:.3f}")
    print(f"  Median |Δr| after partial: {res['delta'].median():.3f}")

if len(results) == 0:
    raise RuntimeError("No modality data loaded. Set AIRBORNE_CSV / STRUCTURE_CSV above.")

## 2 · Duration ↔ depth coupling per modality

In [ ]:
n_mod = len(results)
fig, axes = plt.subplots(1, n_mod, figsize=(6 * n_mod, 4), squeeze=False)

for i, (label, data) in enumerate(results.items()):
    ax = axes[0, i]
    df = data["df"]
    r = data["partial"].iloc[0]["dur_depth_r"]
    sns.boxplot(data=df, x="depth_mm", y="duration_s", ax=ax)
    ax.set_title(f"{label}\nduration ↔ depth  r = {r:.3f}")
    ax.set_xlabel("depth (mm)")
    ax.set_ylabel("duration (s)")

plt.tight_layout()
plt.savefig("validation_crossmod_duration_by_depth.png")
plt.show()

## 3 · Raw vs partial correlation scatter

In [ ]:
fig, axes = plt.subplots(1, n_mod, figsize=(6 * n_mod, 5.5), squeeze=False)

for i, (label, data) in enumerate(results.items()):
    ax = axes[0, i]
    p = data["partial"]
    ax.scatter(p["abs_raw_r"], p["abs_partial_r"], alpha=0.4, s=12)
    mx = max(p["abs_raw_r"].max(), p["abs_partial_r"].max())
    ax.plot([0, mx], [0, mx], "--", linewidth=0.8, color="gray", label="y = x")
    ax.axhline(0.15, ls="--", lw=0.8, color="C1", label="partial |r| = 0.15")
    ax.axhline(0.20, ls=":", lw=0.8, color="C2", label="partial |r| = 0.20")
    ax.set_xlabel("Raw |Spearman r| with depth")
    ax.set_ylabel("Partial |r| (controlling duration)")
    ax.set_title(f"{label}")
    ax.legend(fontsize=7)

plt.suptitle("Raw vs duration-partialled depth signal", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("validation_crossmod_raw_vs_partial.png", bbox_inches="tight")
plt.show()

## 4 · How much depth signal is lost to duration

In [ ]:
fig, axes = plt.subplots(1, n_mod, figsize=(6 * n_mod, 4), squeeze=False)

for i, (label, data) in enumerate(results.items()):
    ax = axes[0, i]
    p = data["partial"]
    ax.hist(p["delta"].dropna(), bins=50, alpha=0.8)
    ax.axvline(0.0, ls="--", lw=0.8)
    med = p["delta"].median()
    ax.axvline(med, ls="-", lw=1.2, color="C3", label=f"median = {med:.3f}")
    ax.set_xlabel("Drop in |r| after partialling")
    ax.set_ylabel("Number of features")
    ax.set_title(f"{label}")
    ax.legend(fontsize=8)

plt.suptitle("Distribution of depth-signal loss after controlling for duration", y=1.02)
plt.tight_layout()
plt.savefig("validation_crossmod_delta_hist.png", bbox_inches="tight")
plt.show()

## 5 · Duration coupling strength vs depth-signal loss

In [ ]:
fig, axes = plt.subplots(1, n_mod, figsize=(6 * n_mod, 5), squeeze=False)

for i, (label, data) in enumerate(results.items()):
    ax = axes[0, i]
    p = data["partial"]
    ax.scatter(p["abs_r_duration"], p["delta"], alpha=0.4, s=12)
    ax.axvline(0.30, ls="--", lw=0.8, label="|r(feat, dur)| = 0.30")
    ax.set_xlabel("|Spearman r| with duration")
    ax.set_ylabel("Drop in |r| after partialling")
    ax.set_title(f"{label}")
    ax.legend(fontsize=8)

plt.suptitle("Features more coupled to duration lose more depth signal", y=1.02)
plt.tight_layout()
plt.savefig("validation_crossmod_coupling_vs_drop.png", bbox_inches="tight")
plt.show()

## 6 · Per-family survival rates

Which feature families retain genuine depth signal after partialling?
This is the most actionable diagnostic: families with high survival rates
are the ones worth investing in; families with near-zero survival are
duration proxies regardless of how many variants you create.

In [ ]:
family_rows = []
for label, data in results.items():
    p = data["partial"]
    for fam, grp in p.groupby("family"):
        family_rows.append(
            {
                "modality": label,
                "family": fam,
                "n_features": len(grp),
                "n_pass_015": grp["survived_015"].sum(),
                "survival_rate_015": grp["survived_015"].mean(),
                "n_pass_020": grp["survived_020"].sum(),
                "survival_rate_020": grp["survived_020"].mean(),
                "median_abs_partial_r": grp["abs_partial_r"].median(),
                "best_partial_r": grp["abs_partial_r"].max(),
                "median_delta": grp["delta"].median(),
            }
        )

fam_df = pd.DataFrame(family_rows).sort_values(
    ["modality", "survival_rate_015"], ascending=[True, False]
)

for label in results:
    print(f"\n{'=' * 70}")
    print(f"  {label.upper()} — family-level duration-dependency summary")
    print(f"{'=' * 70}")
    sub = fam_df[fam_df["modality"] == label]
    print(
        sub[
            [
                "family",
                "n_features",
                "n_pass_015",
                "survival_rate_015",
                "best_partial_r",
                "median_delta",
            ]
        ].to_string(index=False)
    )

# Visual comparison
if len(results) == 2:
    labels = list(results.keys())
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, label in zip(axes, labels):
        sub = fam_df[fam_df["modality"] == label].sort_values("survival_rate_015")
        ax.barh(sub["family"], sub["survival_rate_015"])
        ax.set_xlabel("Fraction surviving |partial r| > 0.15")
        ax.set_title(f"{label}")
        ax.set_xlim(0, 1)
    plt.suptitle("Per-family survival rate after duration partialling", y=1.02)
    plt.tight_layout()
    plt.savefig("validation_crossmod_family_survival.png", bbox_inches="tight")
    plt.show()

## 7 · Top 15 survivors per modality (dumbbell chart)

In [ ]:
fig, axes = plt.subplots(1, n_mod, figsize=(7 * n_mod, 6), squeeze=False)

for i, (label, data) in enumerate(results.items()):
    ax = axes[0, i]
    top = data["partial"].head(15).iloc[::-1]
    ypos = np.arange(len(top))
    for y, raw_v, part_v in zip(ypos, top["abs_raw_r"], top["abs_partial_r"]):
        ax.plot([part_v, raw_v], [y, y], linewidth=1.5)
    ax.scatter(top["abs_partial_r"], ypos, s=24, zorder=5, label="partial |r|")
    ax.scatter(top["abs_raw_r"], ypos, s=24, zorder=5, label="raw |r|")
    ax.axvline(0.15, ls="--", lw=0.8)
    ax.axvline(0.20, ls=":", lw=0.8)
    ax.set_yticks(ypos)
    ax.set_yticklabels(top["feature"], fontsize=7)
    ax.set_xlabel("|r|")
    ax.set_title(f"{label} — top 15 by partial |r|")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig("validation_crossmod_dumbbell_top15.png", bbox_inches="tight")
plt.show()

## 8 · Cross-modality summary

The headline comparison: which modality has genuine depth signal
and which is mostly riding on duration?

In [ ]:
print("\n" + "=" * 75)
print("CROSS-MODALITY DURATION-DEPENDENCY COMPARISON")
print("=" * 75)

header = f"  {'Metric':<50s}"
for label in results:
    header += f"  {label:>10s}"
print(header)
print("  " + "-" * (50 + 12 * len(results)))

metrics = [
    ("Duration ↔ depth Spearman r", lambda p: f"{p.iloc[0]['dur_depth_r']:.3f}"),
    ("Total features", lambda p: f"{len(p)}"),
    ("Features with raw |r| > 0.30", lambda p: f"{(p['abs_raw_r'] > 0.30).sum()}"),
    ("Features with |partial r| > 0.15", lambda p: f"{p['survived_015'].sum()}"),
    ("Features with |partial r| > 0.20", lambda p: f"{p['survived_020'].sum()}"),
    ("Survival rate (partial > 0.15 / total)", lambda p: f"{p['survived_015'].mean():.1%}"),
    ("Best partial |r|", lambda p: f"{p['abs_partial_r'].max():.3f}"),
    ("Median drop in |r| after partialling", lambda p: f"{p['delta'].median():.3f}"),
    ("Mean drop in |r| after partialling", lambda p: f"{p['delta'].mean():.3f}"),
    (
        "Features where partial > raw (gained signal)",
        lambda p: f"{(p['abs_partial_r'] > p['abs_raw_r']).sum()}",
    ),
    ("Best raw feature", lambda p: p.sort_values("abs_raw_r", ascending=False).iloc[0]["feature"]),
    ("Best partial feature", lambda p: p.iloc[0]["feature"]),
]

for name, fn in metrics:
    row = f"  {name:<50s}"
    for label in results:
        row += f"  {fn(results[label]['partial']):>10s}"
    print(row)

print()
for label in results:
    p = results[label]["partial"]
    n = len(p)
    n15 = p["survived_015"].sum()
    dur_r = p.iloc[0]["dur_depth_r"]
    if dur_r > 0.90 and n15 == 0:
        print(f"  ⚠  {label}: ALL features are duration proxies (none survive partialling).")
        print("     The feature set cannot predict depth beyond just exploiting time.")
    elif dur_r > 0.90 and n15 / n < 0.05:
        print(
            f"  ⚠  {label}: severe duration dependency. Only {n15}/{n} features "
            f"({100 * n15 / n:.1f}%) carry genuine depth signal."
        )
    elif n15 / n > 0.10:
        print(
            f"  ✓  {label}: {n15}/{n} features ({100 * n15 / n:.1f}%) survive partialling. "
            f"Genuine process signal present."
        )
    else:
        print(f"  ~  {label}: moderate duration dependency. {n15}/{n} survive.")

# Save combined partialling results
all_partial = pd.concat([data["partial"] for data in results.values()], ignore_index=True)
all_partial.to_csv("validation_crossmod_partialling.csv", index=False)
fam_df.to_csv("validation_crossmod_family_summary.csv", index=False)
print("\nSaved: validation_crossmod_partialling.csv, validation_crossmod_family_summary.csv")